# regulation-graph-reasoning — end-to-end demo

This notebook loads the artifacts produced by `scripts/01..09_*.py` and reproduces the headline figures and tables without retraining.

Pre-requisite (one-time):
```bash
python scripts/01_bootstrap_data.py
python scripts/03_build_baseline.py
python scripts/04_extract_answer_objects.py
python scripts/05_fit_calibration_conformal.py
python scripts/06_run_causal_experiments.py
python scripts/07_build_hypergraph_manifold.py
python scripts/08_run_stress_tests.py
python scripts/09_render_figures.py
```

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np, pandas as pd, polars as pl
from regreason.config import ARTIFACTS, FIGS

## 1. Headline metrics (baseline vs proposed)

In [ ]:
conformal_meta = json.loads((ARTIFACTS / 'conformal.json').read_text())
print(json.dumps(conformal_meta, indent=2))

## 2. Causal results

In [ ]:
causal = json.loads((ARTIFACTS / 'causal_results.json').read_text())
rows = []
for e in causal['estimates']:
    rows.append({'name': e['name'][:55], 'ate': e['ate'],
                 'CI': f"[{e['ate_ci_low']:.3f}, {e['ate_ci_high']:.3f}]",
                 'placebo_ate': e['refutations'].get('placebo_treatment_ate'),
                 'subset_ate': e['refutations'].get('data_subset_ate')})
pd.DataFrame(rows)

## 3. Conformal calibration & DAG

In [ ]:
from IPython.display import IFrame
IFrame(str(FIGS / 'coverage_vs_alpha.pdf'), width=600, height=400)

## 4. Hypergraph manifold

In [ ]:
manifold_meta = json.loads((ARTIFACTS / 'manifold_meta.json').read_text())
print(json.dumps(manifold_meta, indent=2))
IFrame(str(FIGS / 'manifold.pdf'), width=600, height=460)

## 5. Stress tests

In [ ]:
stress = json.loads((ARTIFACTS / 'stress_results.json').read_text())
pd.DataFrame(stress).T